# Session 2: why a small change in a vessel's width is a big change in pressure
**A. Real-world problem.** The body can change the width of its small vessels in seconds.
A change of only a few percent in radius changes how hard the heart must push. Why is the
effect so large? And how do heart rate, the volume pumped per beat and the vessel width
combine into one blood-pressure number?

*This is general education. It is not medical advice and the model is not a diagnostic
tool. All values are typical textbook figures, not a norm for any person.*

**Plan.** (1) resistance of one pipe (Poiseuille), (2) cardiac output = heart rate x stroke
volume, (3) mean arterial pressure = output x total resistance, (4) a design problem.

In [ ]:
import pathlib

import numpy as np

OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

PA_PER_MMHG = 133.322       # 1 mmHg in pascals
MU_BLOOD = 3.5e-3           # Pa s: a typical textbook viscosity of blood (treated as constant)
TPR_TYPICAL = 18.6          # mmHg per (L/min): typical 93 mmHg / 5 L/min

## Step 1: resistance of one pipe
For slow, smooth (laminar) flow of a thick fluid in a straight round tube of length `L` and
radius `r`, Poiseuille's law gives the resistance

    R = 8 mu L / (pi r^4)          (SI units: Pa s per m^3)

The radius is raised to the *fourth* power: half the radius means 2^4 = 16 times the
resistance. Doctors read pressures in mmHg and flows in mL/s, so we convert:
1 mmHg = 133.322 Pa and 1 mL/s = 1e-6 m^3/s.

In [ ]:
def poiseuille_R(mu, length, radius):
    """Resistance in Pa s / m^3 of a straight round tube (mu in Pa s, length and radius in m)."""
    if radius <= 0 or length <= 0 or mu <= 0:
        raise ValueError("mu, length and radius must be positive")
    return 8.0 * mu * length / (np.pi * radius ** 4)


def to_mmHg_per_mLs(R_si):
    """Convert Pa s / m^3 into mmHg per (mL/s)."""
    return R_si * 1e-6 / PA_PER_MMHG


def resistance_factor(radius_ratio):
    """How many times the resistance grows when the radius becomes radius_ratio times as big."""
    return radius_ratio ** -4


def radius_ratio_for(resistance_ratio):
    """Radius ratio that multiplies the resistance by resistance_ratio (inverse of the above)."""
    return resistance_ratio ** -0.25

## Step 2: cardiac output
Each beat pushes out a *stroke volume* `SV` (mL). With `HR` beats per minute the heart
delivers the *cardiac output* `CO = HR x SV`, in mL per minute; divide by 1000 for L/min.
Typical textbook values: 70 beats per minute and 70 mL give 4.9, about 5 L/min.

In [ ]:
def cardiac_output(hr, sv_mL):
    """Cardiac output in L/min from heart rate (beats/min) and stroke volume (mL)."""
    return hr * sv_mL / 1000.0

## Step 3: mean arterial pressure
The flow through the whole body obeys `Q = dP / R`. Taking the pressure of the returning
blood as about 0, the average pressure in the arteries is

    MAP = CO x TPR

where TPR is the total peripheral resistance. Typical textbook values: MAP about 93 mmHg and
CO about 5 L/min give TPR = 18.6 mmHg per (L/min). A common rule of thumb for MAP from a
reading "systolic/diastolic" is diastolic + (systolic - diastolic)/3; for 120/80 it is 93.3.

In [ ]:
def map_from_co(co, tpr):
    return co * tpr


def tpr_from(map_mmHg, co):
    if co <= 0:
        raise ValueError("cardiac output must be positive")
    return map_mmHg / co


def map_from_bp(systolic, diastolic):
    """Rule-of-thumb mean arterial pressure from a systolic/diastolic reading (mmHg)."""
    return diastolic + (systolic - diastolic) / 3.0


def map_model(hr, sv_mL, radius_ratio=1.0, tpr0=TPR_TYPICAL):
    """MAP (mmHg) when all the small vessels have radius_ratio times their normal radius."""
    return map_from_co(cardiac_output(hr, sv_mL), tpr0 * resistance_factor(radius_ratio))

## Step 4: see it
Left: MAP against vessel radius at constant output. Right: the resistance factor 1/r^4.

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    R1 = poiseuille_R(MU_BLOOD, 0.10, 1.0e-3)
    R2 = poiseuille_R(MU_BLOOD, 0.10, 0.5e-3)
    print("R (r = 1.0 mm):", round(to_mmHg_per_mLs(R1), 3), "mmHg per mL/s")
    print("R (r = 0.5 mm):", round(to_mmHg_per_mLs(R2), 2), "mmHg per mL/s; ratio", round(R2 / R1, 3))
    print("CO:", cardiac_output(70, 70), "L/min; TPR from 93 and 5:", tpr_from(93, 5))
    print("MAP +20% resistance at constant CO:", round(map_from_co(5.0, TPR_TYPICAL * 1.2), 1), "mmHg")
    print("radius change that gives +20% resistance:", round(radius_ratio_for(1.2), 4))
    r = np.linspace(0.8, 1.2, 81)
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(r, [map_from_co(5.0, TPR_TYPICAL * resistance_factor(x)) for x in r])
    ax.axhline(93, color="gray", ls=":")
    ax.set_xlabel("vessel radius / normal radius")
    ax.set_ylabel("mean arterial pressure at 5 L/min (mmHg)")
    ax.set_title("A few percent in radius moves the pressure a lot")
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(OUT / "map_vs_radius.png", dpi=120)

    rr = np.linspace(0.3, 1.5, 100)
    fig2, ax2 = plt.subplots(figsize=(6, 4))
    ax2.semilogy(rr, resistance_factor(rr))
    ax2.axvline(0.5, color="gray", ls=":")
    ax2.set_xlabel("radius / normal radius")
    ax2.set_ylabel("resistance / normal resistance (log scale)")
    ax2.set_title("R grows as 1 / r^4: half the radius is 16 times the resistance")
    ax2.grid(alpha=0.3)
    fig2.tight_layout()
    fig2.savefig(OUT / "resistance_vs_radius.png", dpi=120)
    print("saved figures in", OUT)

## Try it
Do the coding tasks, answer the four questions in the README (section F), then run the
checkpoint.

## Coding tasks
1. **Easy.** Use `poiseuille_R` and `to_mmHg_per_mLs` to find the resistance of the same vessel for radii 1.0, 0.8 and 0.5 mm and check that the ratios are 1, 2.44 and 16.
2. **Stretch.** Write `co_for_map(map_target, radius_ratio)` that returns the cardiac output needed to give a target MAP at a given vessel radius, and use it for the design problem in Section F.
3. **Explain.** Explain why a 4.5 percent narrowing of the radius has the same effect on resistance as a 20 percent rise, and why this makes the circulation sensitive to small changes in vessel width.

## Assessment
- **A. Conceptual.** Why does the resistance depend on the radius to the fourth power and not just on the cross-section area (radius squared)?
- **B. Numerical.** A heart beats 80 times per minute with a stroke volume of 60 mL. Find the cardiac output, and the MAP for a TPR of 18.6 mmHg per (L/min). Check the units.
- **C. Graph and simulation interpretation.** In the graph of MAP against radius at 5 L/min, read the MAP at radius 0.9 and 1.1 of normal. Why are the two changes from 93 not equal?
- **D. Design and optimisation challenge.** Design: the vessels are 5 percent narrower (radius 0.95 of normal) and the mean arterial pressure must stay at or below 100 mmHg. Find the largest cardiac output and choose a heart rate and stroke volume. State the trade-offs.

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    R1 = L.poiseuille_R(L.MU_BLOOD, 0.10, 1.0e-3)
    R2 = L.poiseuille_R(L.MU_BLOOD, 0.10, 0.5e-3)
    checks.append(("worked example: half the radius gives 16 times the resistance", abs(R2 / R1 - 16.0) < 1e-9))
    checks.append(("worked example: r = 1.0 mm, L = 0.10 m gives 6.685 mmHg per (mL/s)",
                   abs(L.to_mmHg_per_mLs(R1) - 6.685) < 1e-3))
    checks.append(("worked example: r = 0.5 mm gives 106.96 mmHg per (mL/s)", abs(L.to_mmHg_per_mLs(R2) - 106.96) < 1e-2))
    checks.append(("a 10 mmHg push through the 1.0 mm tube gives 1.496 mL/s",
                   abs(10.0 / L.to_mmHg_per_mLs(R1) - 1.4959) < 1e-3))
    checks.append(("hand calculation: R = 8 mu L / (pi r^4) = 8.913e8 Pa s/m^3 for r = 1 mm",
                   abs(R1 - 8 * 3.5e-3 * 0.1 / (np.pi * 1e-12)) < 1.0 and abs(R1 / 1e8 - 8.913) < 1e-3))
    checks.append(("cardiac output 70 x 70 mL is 4.9 L/min", abs(L.cardiac_output(70, 70) - 4.9) < 1e-12))
    checks.append(("worked example: TPR = 93 / 5 = 18.6 mmHg per (L/min)", abs(L.tpr_from(93.0, 5.0) - 18.6) < 1e-12))
    checks.append(("rule of thumb: 120/80 gives a mean pressure of 93.3 mmHg", abs(L.map_from_bp(120, 80) - 93.333) < 1e-3))
    checks.append(("worked example: +20% resistance at 5 L/min gives 111.6 mmHg",
                   abs(L.map_from_co(5.0, 1.2 * L.TPR_TYPICAL) - 111.6) < 1e-9))
    checks.append(("the radius that gives +20% resistance is 0.9554 of normal (4.5% narrower)",
                   abs(L.radius_ratio_for(1.2) - 0.9554) < 1e-4 and abs(L.resistance_factor(L.radius_ratio_for(1.2)) - 1.2) < 1e-12))
    checks.append(("radius 5% smaller gives 1.228 times the resistance", abs(L.resistance_factor(0.95) - 1.2277) < 1e-4))
    checks.append(("typical heart (70 x 70 mL), normal vessels: MAP 91.1 mmHg", abs(L.map_model(70, 70) - 91.14) < 1e-9))
    checks.append(("radius 5% smaller at 70 x 70 mL: MAP 111.9 mmHg", abs(L.map_model(70, 70, 0.95) - 111.896) < 1e-2))
    checks.append(("design: 60 bpm x 70 mL with radius 0.95 keeps MAP at 95.9 mmHg (below 100)",
                   abs(L.map_model(60, 70, 0.95) - 95.911) < 1e-2 and L.map_model(60, 70, 0.95) < 100.0))
    checks.append(("halving the radius at constant 4.9 L/min would need 1458 mmHg: impossible",
                   abs(L.map_model(70, 70, 0.5) - 1458.24) < 1e-2))
    checks.append(("radius ratio 1 changes nothing", abs(L.resistance_factor(1.0) - 1.0) < 1e-15))
    checks.append(("resistance and radius ratios are inverse to each other",
                   abs(L.radius_ratio_for(L.resistance_factor(0.7)) - 0.7) < 1e-12))
    bad = 0
    for args in ((0, 1, 1), (1, 0, 1), (1, 1, 0)):
        try:
            L.poiseuille_R(*args)
        except ValueError:
            bad += 1
    checks.append(("zero viscosity, length or radius is rejected", bad == 3))
    try:
        L.tpr_from(93.0, 0.0)
        ok0 = False
    except ValueError:
        ok0 = True
    checks.append(("zero cardiac output is rejected instead of dividing by zero", ok0))

    # --- numbers used in the text
    checks.append(("worked example: 8 mu L = 2.8e-3 and pi r^4 = 3.142e-12", abs(8 * L.MU_BLOOD * 0.10 - 2.8e-3) < 1e-12 and abs(np.pi * 1e-12 - 3.142e-12) < 1e-15))
    checks.append(("worked example: the flow for 10 mmHg through the 0.5 mm tube is 0.0935 mL/s", abs(10.0 / L.to_mmHg_per_mLs(R2) - 0.0935) < 1e-4))
    m09 = L.map_from_co(5.0, L.TPR_TYPICAL * L.resistance_factor(0.9))
    m11 = L.map_from_co(5.0, L.TPR_TYPICAL * L.resistance_factor(1.1))
    checks.append(("MAP at 5 L/min is 141.7 at radius 0.9 and 63.5 at radius 1.1 (changes +48.7 and -29.5)",
                   abs(m09 - 141.74) < 0.01 and abs(m11 - 63.52) < 0.01 and abs(m09 - 93 - 48.74) < 0.01 and abs(93 - m11 - 29.48) < 0.01))
    checks.append(("factors 1.524 (r = 0.9), 0.683 (r = 1.1) and 2.44 (r = 0.8)",
                   abs(L.resistance_factor(0.9) - 1.5242) < 1e-4 and abs(L.resistance_factor(1.1) - 0.6830) < 1e-4 and abs(L.resistance_factor(0.8) - 2.4414) < 1e-4))
    checks.append(("80 beats x 60 mL is 4.8 L/min and MAP 89.3", abs(L.cardiac_output(80, 60) - 4.8) < 1e-12 and abs(L.map_from_co(4.8, 18.6) - 89.28) < 1e-9))
    tpr_n = 18.6 * L.resistance_factor(0.95)
    checks.append(("design: TPR 22.8 and the largest output for MAP 100 is 4.38 L/min", abs(tpr_n - 22.84) < 0.01 and abs(100.0 / tpr_n - 4.379) < 1e-3))
    checks.append(("a 5 L/min case from 100 x 50 mL gives exactly 93 mmHg", abs(L.map_model(100, 50) - 93.0) < 1e-9))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
